In [ ]:
# ================================================================
# COMPLETE COLAB SCRIPT: RankRAG-inspired + LLM Fake News Detection
# Upload a labelled CSV when prompted.
# Labels: Fake=0, Real=1 (edit LABEL_MAP if your dataset differs).
# This is a RankRAG-inspired research implementation, not official RankRAG.
# ================================================================

# 1) Install dependencies (run inside Google Colab)
import sys, subprocess, os, re, io, json, random, warnings
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "accelerate", "peft", "trl", "sentence-transformers",
    "scikit-learn", "pandas", "bitsandbytes", "sentencepiece"
])

import numpy as np
import pandas as pd
import torch
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)
from sentence_transformers import SentenceTransformer, CrossEncoder
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ------------------- CONFIGURATION -------------------
SEED = 42
TEXT_COL = "Body"       # UPDATED to match your CSV's text column
LABEL_COL = "Label"     # UPDATED to match your CSV's label column

# Label mapping: Fake=0, Real=1. Edit after inspecting your CSV labels.
LABEL_MAP = {
    "fake": 0, "false": 0, "rumor": 0, "rumour": 0, "0": 0,
    "real": 1, "true": 1, "non-rumor": 1, "non_rumor": 1, "1": 1
}

# Qwen is an accessible alternative to gated Llama 3.1.
# To use Llama 3.1, set to "meta-llama/Llama-3.1-8B-Instruct"
# and ensure your Hugging Face account has accepted Meta's terms.
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RERANKER_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

TOP_CANDIDATES = 20
TOP_K = 4
MAX_LENGTH = 1024
EPOCHS = 1
MAX_TRAIN_ROWS = None   # Set to 100 for a quick smoke test; None uses all training rows
MAX_TEST_ROWS = 100     # Set to None to evaluate the complete test split
OUTPUT_DIR = "/content/rankrag_fake_news_output"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("=" * 70)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Select Runtime > Change runtime type > GPU in Colab.")

# ------------------- LOAD DATA -------------------
print("\nUpload your labelled CSV file.")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No CSV uploaded.")
csv_name = next(iter(uploaded))
raw_df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("Loaded:", csv_name, "| shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
print("Raw label counts:")
print(raw_df[LABEL_COL].value_counts(dropna=False) if LABEL_COL in raw_df.columns
      else "LABEL_COL not found; edit LABEL_COL in the configuration section.")

if TEXT_COL not in raw_df.columns or LABEL_COL not in raw_df.columns:
    raise ValueError(
        f"Set TEXT_COL and LABEL_COL correctly. Available columns: {raw_df.columns.tolist()}"
    )

def map_label(value):
    key = str(value).strip().lower()
    if key not in LABEL_MAP:
        raise ValueError(
            f"Unmapped label {value!r}. Edit LABEL_MAP. "
            f"Observed values: {raw_df[LABEL_COL].astype(str).unique()[:30].tolist()}"
        )
    return LABEL_MAP[key]

df = raw_df[[TEXT_COL, LABEL_COL]].rename(
    columns={TEXT_COL: "text", LABEL_COL: "raw_label"}
).copy()
df = df.dropna(subset=["text", "raw_label"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0]
df["label"] = df["raw_label"].apply(map_label).astype(int)
df = df[["text", "label"]].drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

if df["label"].nunique() != 2:
    raise ValueError("After label mapping, the dataset must contain both classes 0 and 1.")
if df["label"].value_counts().min() < 3:
    raise ValueError("Need at least 3 examples in each class to make train/validation/test splits.")

print("\nPrepared data:", df.shape)
print("Label counts (0=Fake, 1=Real):", df["label"].value_counts().sort_index().to_dict())

train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.1765, stratify=train_val_df["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

if MAX_TRAIN_ROWS is not None:
    train_df = train_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)

print("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))

# ------------------- RETRIEVAL + RERANKING -------------------
print("\nLoading multilingual retrieval and reranking models...")
embedder = SentenceTransformer(EMBED_MODEL_NAME)
reranker = CrossEncoder(RERANKER_NAME, max_length=256)

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].astype(int).tolist()
train_embeddings = embedder.encode(
    train_texts, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

def retrieve_and_rerank(query, top_candidates=TOP_CANDIDATES, top_k=TOP_K,
                        exclude_index=None):
    q_emb = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    similarities = train_embeddings @ q_emb
    if exclude_index is not None and 0 <= exclude_index < len(similarities):
        similarities[exclude_index] = -1e9
    candidate_ids = np.argsort(similarities)[::-1][:min(top_candidates, len(train_texts))]
    if len(candidate_ids) == 0:
        return []
    candidate_texts = [train_texts[int(i)] for i in candidate_ids]
    pair_scores = reranker.predict(
        [(query, candidate) for candidate in candidate_texts],
        show_progress_bar=False
    )
    order = np.argsort(pair_scores)[::-1][:min(top_k, len(candidate_ids))]
    results = []
    for rank, j in enumerate(order, start=1):
        idx = int(candidate_ids[j])
        results.append({
            "rank": rank,
            "text": train_texts[idx],
            "label": int(train_labels[idx]),
            "similarity": float(similarities[idx]),
            "rerank_score": float(pair_scores[j])
        })
    return results

def build_prompt(text, contexts):
    lines = []
    for i, item in enumerate(contexts, 1):
        label_name = "Fake" if int(item["label"]) == 0 else "Real"
        lines.append(
            f"Similar labelled training example {i} (dataset label: {label_name}): "
            f"{item['text']}"
        )
    context_text = "\n".join(lines) if lines else "No similar examples retrieved."
    return f"""You are a careful fake-news classification assistant.
Classify the post as Fake or Real using the post and the similar labelled training examples.
The examples are dataset-labelled similar examples, not independently verified evidence.
Do not claim external verification. Give one short reason and acknowledge uncertainty if appropriate.

SIMILAR TRAINING EXAMPLES:
{context_text}

POST:
{text}

Respond in this format:
Label: Fake or Real
Reason: one short sentence
"""

# ------------------- PREPARE INSTRUCTION DATA -------------------
print("\nBuilding retrieval-augmented training prompts...")
train_records = []
for i, row in train_df.iterrows():
    contexts = retrieve_and_rerank(row["text"], exclude_index=i)
    prompt = build_prompt(row["text"], contexts)
    target = "Fake" if int(row["label"]) == 0 else "Real"
    train_records.append({
        "text": prompt + f"\nLabel: {target}\nReason: Training label supplied by the dataset."
    })
    if (i + 1) % 100 == 0:
        print(f"Prepared {i + 1}/{len(train_df)} training examples")

val_records = []
for _, row in val_df.iterrows():
    contexts = retrieve_and_rerank(row["text"])
    prompt = build_prompt(row["text"], contexts)
    target = "Fake" if int(row["label"]) == 0 else "Real"
    val_records.append({
        "text": prompt + f"\nLabel: {target}\nReason: Training label supplied by the dataset."
    })

print("Instruction examples:", len(train_records), "train;", len(val_records), "validation")

# ------------------- LOAD BASE LLM + LoRA -------------------
print("\nLoading language model:", BASE_MODEL)
if "llama-3.1" in BASE_MODEL.lower():
    print("Reminder: Llama 3.1 requires accepted model terms and authorized Hugging Face access.")
    try:
        from huggingface_hub import notebook_login
        notebook_login()
    except Exception as e:
        print("If access fails, switch BASE_MODEL to Qwen/Qwen2.5-3B-Instruct.", e)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = None
if torch.cuda.is_available():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules="all-linear"
)

train_hf = Dataset.from_list(train_records)
val_hf = Dataset.from_list(val_records)

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_length=MAX_LENGTH,
    packing=False,
    report_to="none",
    seed=SEED
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    peft_config=lora_config,
    processing_class=tokenizer
)
trainer.train()

# ------------------- HELD-OUT TEST -------------------
def parse_label(generated):
    matches = re.findall(r"(?im)^\s*Label\s*:\s*(Fake|Real)\b", generated)
    if matches:
        return 0 if matches[-1].lower() == "fake" else 1
    m = re.search(r"(?i)\b(Fake|Real)\b", generated)
    if m:
        return 0 if m.group(1).lower() == "fake" else 1
    return None

def generate_answer(prompt, max_new_tokens=80):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    continuation = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(continuation, skip_special_tokens=True)

eval_df = test_df if MAX_TEST_ROWS is None else test_df.iloc[:MAX_TEST_ROWS].copy()
prediction_rows = []
print("\nPredicting held-out test examples...")
for i, row in eval_df.reset_index(drop=True).iterrows():
    contexts = retrieve_and_rerank(row["text"])
    prompt = build_prompt(row["text"], contexts)
    answer = generate_answer(prompt)
    predicted = parse_label(answer)
    prediction_rows.append({
        "text": row["text"],
        "true_label": int(row["label"]),
        "predicted_label": predicted,
        "true_class": "Fake" if int(row["label"]) == 0 else "Real",
        "predicted_class": None if predicted is None else ("Fake" if predicted == 0 else "Real"),
        "generated_answer": answer
    })
    if (i + 1) % 10 == 0:
        print(f"Predicted {i+1}/{len(eval_df)}")

pred_df = pd.DataFrame(prediction_rows)
valid_df = pred_df.dropna(subset=["predicted_label"]).copy()
coverage = len(valid_df) / max(1, len(pred_df))

print("\n" + "=" * 70)
print("TEST RESULTS")
print("Prediction coverage:", round(coverage, 4), f"({len(valid_df)}/{len(pred_df)})")
if len(valid_df):
    y_true = valid_df["true_label"].astype(int)
    y_pred = valid_df["predicted_label"].astype(int)
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )[2], 4))
    print("\nClassification report (0=Fake, 1=Real):")
    print(classification_report(
        y_true, y_pred, labels=[0, 1], target_names=["Fake", "Real"],
        digits=4, zero_division=0
    ))
    print("Confusion matrix (rows=true, columns=predicted; Fake, Real):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
else:
    print("No valid predictions parsed. Inspect generated_answer and adjust the prompt/parser.")

# ------------------- SAVE OUTPUTS -------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)
pred_path = "/content/rankrag_test_predictions.csv"
summary_path = "/content/rankrag_experiment_summary.json"
pred_df.to_csv(pred_path, index=False)

summary = {
    "implementation": "RankRAG-inspired retrieval + reranking + LoRA instruction fine-tuning",
    "base_model": BASE_MODEL,
    "embedding_model": EMBED_MODEL_NAME,
    "reranker": RERANKER_NAME,
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows_evaluated": int(len(eval_df)),
    "prediction_coverage": float(coverage),
    "seed": SEED,
    "top_candidates": TOP_CANDIDATES,
    "top_k": TOP_K
}
if len(valid_df):
    summary["accuracy"] = float(accuracy_score(
        valid_df["true_label"].astype(int), valid_df["predicted_label"].astype(int)
    ))
    summary["macro_f1"] = float(precision_recall_fscore_support(
        valid_df["true_label"].astype(int), valid_df["predicted_label"].astype(int),
        average="macro", zero_division=0
    )[2])
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# Save LoRA adapter and tokenizer
adapter_path = "/content/rankrag_lora_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
zip_path = "/content/rankrag_lora_adapter.zip"
subprocess.run(["zip", "-qr", zip_path, adapter_path], check=False)

print("\nSaved:")
print(pred_path)
print(summary_path)
print(zip_path)
print(json.dumps(summary, indent=2))

print("\nDownload outputs now.")
files.download(pred_path)
files.download(summary_path)
files.download(zip_path)


GPU available: False

Upload your labelled CSV file.
